In [5]:
import pandas as pd
import numpy as np
import re
import os
import shutil
from datetime import datetime, time
from IPython.display import display

RUTA_ARCHIVO_ORIGEN = 'Programa_32IIT_21-09.xlsx'
CARPETA_SALIDA = 'output_db'

# Clases (Etapas)
class ETLDataLoader:
    def __init__(self, filepath):
        self.filepath = filepath
        self.df = None

    def load_data(self):
        if not os.path.exists(self.filepath):
            raise FileNotFoundError(f"Error crítico: El archivo {self.filepath} no existe en la ruta especificada.")
        try:
            if self.filepath.endswith('.csv'):
                self.df = pd.read_csv(self.filepath, encoding='utf-8')
            elif self.filepath.endswith('.xlsx'):
                self.df = pd.read_excel(self.filepath, skiprows=8)
            else:
                raise ValueError("Formato no soportado. Usa CSV o XLSX.")

            if self.df.empty:
                raise ValueError("Error crítico: El archivo está vacío.")
            print(f"[OK] Carga exitosa. Registros leidos: {len(self.df)}")
            return self.df
        except Exception as e:
            raise Exception(f"Error critico al leer el archivo: {str(e)}")


class SchemaValidator:
    def __init__(self, df):
        self.df = df
        self.expected_columns = [
            'Actividad', 'Fecha inicio', 'Fecha fin', 'Hora inicio', 'Hora fin',
            'Tipo', 'Salón', 'Título', 'Ponente', 'Empresa / Institución',
            'Maestro de ceremonias', 'Enlace'
        ]

    def validate_layout(self):
        missing_cols = [col for col in self.expected_columns if col not in self.df.columns]
        if missing_cols:
            raise ValueError(f"Error de Layout: Faltan las columnas {missing_cols}")

        print("[OK] Layout validado: Todas las columnas requeridas estan presentes.")

        try:
            self.df['Fecha inicio'] = pd.to_datetime(self.df['Fecha inicio'], format='%Y-%m-%d', errors='coerce')
            self.df['Fecha fin'] = pd.to_datetime(self.df['Fecha fin'], format='%Y-%m-%d', errors='coerce')
            self.df['Hora inicio'] = pd.to_datetime(self.df['Hora inicio'].astype(str), format='%H:%M', errors='coerce').dt.time
            self.df['Hora fin'] = pd.to_datetime(self.df['Hora fin'].astype(str), format='%H:%M', errors='coerce').dt.time
            print("[OK] Tipos de datos convertidos correctamente (datetime / time).")
        except Exception as e:
            raise Exception(f"Error critico en la tipificacion de datos: {str(e)}")

        return self.df


class DataCleanerAndValidator:
    def __init__(self, df):
        self.df = df.copy()
        self.df_valid = pd.DataFrame()
        self.df_rejected = pd.DataFrame()

        self.actividades_validas = [
            'Conferencia', 'Taller', 'Foro', 'Concurso', 'Ponencia', 'Presentación',
            'Curso', 'Deportiva', 'Cultural', 'Otra', 'Conferencia magistral',
            'Visita de Campo', 'Panel', 'Exposicion', 'Cátedra Patrimonial',
            'Mesa de diálogo', 'Capacitación', 'FOSEEI'
        ]
        self.modalidades_validas = ['presencial', 'virtual', 'híbrida']

    def clean_and_validate(self):
        valid_records = []
        rejected_records = []

        fecha_min = pd.to_datetime('2026-09-20')
        fecha_max = pd.to_datetime('2026-09-25')

        for index, row in self.df.iterrows():
            razones_rechazo = []
            correcciones_aplicadas = []

            # LIMPIEZA
            act_original = str(row['Actividad']).strip()
            if act_original not in self.actividades_validas:
                row['Actividad'] = 'Otra'
                correcciones_aplicadas.append(f"Actividad corregida de '{act_original}' a 'Otra'")

            modalidad = str(row['Tipo']).strip().lower()
            row['Tipo'] = modalidad

            if str(row['Ponente']).strip() == str(row['Maestro de ceremonias']).strip() and pd.notna(row['Ponente']):
                row['Maestro de ceremonias'] = np.nan
                correcciones_aplicadas.append("Rol duplicado: Se priorizó como Ponente")

            # VALIDACIÓN
            if pd.isna(row['Título']) or str(row['Título']).strip() == "":
                razones_rechazo.append("Titulo vacio")
            if pd.isna(row['Fecha inicio']) or pd.isna(row['Fecha fin']):
                razones_rechazo.append("Fechas nulas o invalidas")

            if pd.notna(row['Fecha inicio']) and (row['Fecha inicio'] < fecha_min or row['Fecha inicio'] > fecha_max):
                razones_rechazo.append("Fecha fuera del rango permitido")
            if pd.notna(row['Hora inicio']) and pd.notna(row['Hora fin']):
                if row['Hora inicio'] >= row['Hora fin']:
                    razones_rechazo.append("Hora inicio mayor o igual a Hora fin")

            if modalidad not in self.modalidades_validas:
                razones_rechazo.append(f"Modalidad no valida: {row['Tipo']}")

            salon_vacio = pd.isna(row['Salón']) or str(row['Salón']).strip() == "" or str(row['Salón']).strip().lower() == "nan"
            if modalidad == 'presencial' and salon_vacio:
                razones_rechazo.append("Sesion presencial sin salon asignado")

            enlace = str(row['Enlace']).strip()
            if pd.notna(row['Enlace']) and enlace != "" and enlace.lower() != "nan":
                if not re.match(r'^https://(teams|events\.teams)\.microsoft\.com/', enlace):
                    razones_rechazo.append("Enlace de reunion no pertenece a MS Teams")
            elif modalidad == 'virtual':
                 razones_rechazo.append("Sesion virtual sin enlace de Teams")

            # SEPARACIÓN
            if len(razones_rechazo) > 0:
                row_dict = row.to_dict()
                row_dict['Motivo_Rechazo'] = " | ".join(razones_rechazo)
                rejected_records.append(row_dict)
            else:
                row_dict = row.to_dict()
                row_dict['Correcciones_Aplicadas'] = " | ".join(correcciones_aplicadas) if correcciones_aplicadas else "Ninguna"
                valid_records.append(row_dict)

        self.df_valid = pd.DataFrame(valid_records)
        self.df_rejected = pd.DataFrame(rejected_records)

        print(f"[OK] Limpieza y Validacion completadas.")
        print(f"   - Registros validos: {len(self.df_valid)}")
        print(f"   - Registros rechazados: {len(self.df_rejected)}")
        return self.df_valid, self.df_rejected


class DataTransformer:
    def __init__(self, df_valid):
        self.df = df_valid
        self.counters = {'E': 1, 'S': 1, 'P': 1, 'L': 1, 'D': 1, 'T': 1, 'M': 1}
        self.personas_dict = {}

        self.departamentos_base = {
            'D001': 'Física y Matemáticas',
            'D002': 'Ingeniería Eléctrica y Computación',
            'D003': 'Ingeniería Industrial y Manufactura',
            'D004': 'Ingeniería Civil y Ambiental'
        }
        self.departamentos_map = {
            'Física': 'D001', 'Matemáticas': 'D001',
            'Eléctrica': 'D002', 'Computación': 'D002', 'Software': 'D002', 'IA': 'D002',
            'Industrial': 'D003', 'Manufactura': 'D003', 'Mecatrónica': 'D003',
            'Civil': 'D004', 'Ambiental': 'D004', 'Geociencias': 'D004'
        }

    def _generate_id(self, prefix):
        id_str = f"{prefix}{str(self.counters[prefix]).zfill(3)}"
        self.counters[prefix] += 1
        return id_str

    def _infer_cupo(self, ubicacion, desc_lugar):
        if "MACROAULA" in ubicacion or "AUDITORIO" in ubicacion: return 250
        if "CANCHA" in ubicacion or "EXPLANADA" in ubicacion or "VESTIBULO" in ubicacion: return 100
        if "LAB" in ubicacion: return 30
        return 40

    def _parse_salon(self, salon_str):
        if pd.isna(salon_str) or str(salon_str).strip().lower() == "nan" or str(salon_str).strip() == "":
            return "N/A", "N/A", "N/A"

        salon = str(salon_str).upper()
        campus = "IIT"
        ubicacion = "Aula"

        if "CUAUH" in salon: campus = "CUAUH"
        elif "CU " in salon or salon.startswith("CU"): campus = "CU"

        if "MACROAULA" in salon: ubicacion = "Macroaula"
        elif "AUDIOVISUAL" in salon: ubicacion = "Audiovisual"
        elif "CANCHA" in salon: ubicacion = "Cancha"
        elif "LAB" in salon: ubicacion = "Laboratorio"
        elif "EXPLANADA" in salon or "VESTIBULO" in salon: ubicacion = "Explanada/Vestibulo"

        return campus, ubicacion, salon_str.strip()

    def _assign_department(self, row):
        texto_busqueda = str(row['Título']) + " " + str(row['Empresa / Institución']) + " " + str(row['Ponente'])
        for key, dept_id in self.departamentos_map.items():
            if key.lower() in texto_busqueda.lower():
                return dept_id
        return 'D002'

    def transform(self):
        eventos, sesiones, personas, lugares, presentan, tipos_actividad = [], [], [], [], [], []
        departamentos, empresas, lidera = [], [], []

        for dept_id, nombre in self.departamentos_base.items():
            departamentos.append({'Departamento_id': dept_id, 'Nombre_departamento': nombre})

        tipos_base = [
            'Conferencia', 'Taller', 'Foro', 'Concurso', 'Ponencia', 'Presentación',
            'Curso', 'Deportiva', 'Cultural', 'Otra', 'Conferencia magistral',
            'Visita de Campo', 'Panel', 'Exposicion', 'Cátedra Patrimonial',
            'Mesa de diálogo', 'Capacitación', 'FOSEEI'
        ]
        tipo_dict = {}
        for tipo in tipos_base:
            tid = self._generate_id('T')
            tipo_dict[tipo] = tid
            desc_tipo = "Un foro para difundir temas de tecnología a estudiantes de licenciatura y posgrado" if tipo == 'FOSEEI' else f'Actividad de tipo {tipo}'
            tipos_actividad.append({'Tipo_actividad_id': tid, 'Nombre_tipo': tipo, 'Descripcion_tipo': desc_tipo})

        # CORRECCIÓN DE CALIDAD DE DATOS
        empresas_raw = self.df['Empresa / Institución'].dropna().unique()
        empresa_dict = {}
        empresas_agregadas = set()

        for emp in empresas_raw:
            emp_str = str(emp).strip()
            if emp_str and emp_str.lower() != "nan":
                emp_key = emp_str.lower() # Clave única

                if emp_key not in empresas_agregadas:
                    eid = self._generate_id('M')
                    empresa_dict[emp_key] = eid
                    empresas_agregadas.add(emp_key)
                    empresas.append({'Empresa_id': eid, 'Nombre': emp_str})

        for _, row in self.df.iterrows():
            evento_id = self._generate_id('E')
            depto_id = self._assign_department(row)
            desc_evento = f"{row['Actividad']} enfocado en {str(row['Título']).lower()}. {row['Empresa / Institución']}."

            eventos.append({
                'Evento_id': evento_id, 'Tipo_actividad_id': tipo_dict.get(row['Actividad'], 'T000'),
                'Departamento_id': depto_id, 'Titulo_evento': row['Título'], 'Descripcion_evento': desc_evento
            })

            # Búsqueda del ID de la empresa usando la misma clave
            emp_nombre = str(row['Empresa / Institución']).strip()
            emp_key_busqueda = emp_nombre.lower()

            if emp_key_busqueda in empresa_dict:
                lidera.append({'Evento_id': evento_id, 'Empresa_id': empresa_dict[emp_key_busqueda]})

            fecha_actual = row['Fecha inicio']
            while fecha_actual <= row['Fecha fin']:
                sesion_id = self._generate_id('S')
                lugar_id = self._generate_id('L')

                campus, ubicacion, desc_lugar = self._parse_salon(row['Salón'])
                cupo_inferido = self._infer_cupo(ubicacion, desc_lugar)

                lugares.append({
                    'Lugar_id': lugar_id, 'Descripcion_lugar': desc_lugar,
                    'Cupo': cupo_inferido, 'Ubicacion': ubicacion, 'Campus': campus
                })

                link_reunion = row['Enlace'] if str(row['Tipo']).lower() in ['virtual', 'híbrida'] else np.nan

                ponentes_str = str(row['Ponente'])
                cantidad_ponentes = len(ponentes_str.split(',')) if pd.notna(row['Ponente']) else 1
                es_unica = True if cantidad_ponentes <= 1 else False

                sesiones.append({
                    'Sesion_id': sesion_id, 'Evento_id': evento_id, 'Lugar_id': lugar_id,
                    'Modalidad': str(row['Tipo']).capitalize(), 'Fecha': fecha_actual.date(),
                    'Hora_inicio': row['Hora inicio'], 'Hora_fin': row['Hora fin'],
                    'Link_reunion': link_reunion,
                    'Es_unica': es_unica
                })
                fecha_actual += pd.Timedelta(days=1)

            def procesar_persona(nombre, rol, ev_id):
                if pd.isna(nombre) or str(nombre).strip() == "" or str(nombre).lower() == "nan":
                    return

                nombres_array = str(nombre).split(',')
                for nom in nombres_array:
                    nom = nom.strip()
                    if not nom: continue

                    if nom in self.personas_dict:
                        persona_id = self.personas_dict[nom]
                    else:
                        persona_id = self._generate_id('P')
                        self.personas_dict[nom] = persona_id

                        if "estudiantes" in nom.lower() or "centro" in nom.lower():
                            titulo, p_nombre, s_nombre, p_apellido, s_apellido = "", nom, "", "", ""
                        else:
                            partes = nom.split()
                            titulo = partes[0] if partes[0] in ['Dr.', 'Dra.', 'Mtro.', 'Mtra.', 'Ing.', 'Lic.'] else ""
                            partes_limp = partes[1:] if titulo else partes
                            p_nombre = partes_limp[0] if len(partes_limp) > 0 else ""
                            p_apellido = partes_limp[1] if len(partes_limp) > 1 else ""
                            s_nombre = partes_limp[2] if len(partes_limp) > 2 else ""
                            s_apellido = partes_limp[3] if len(partes_limp) > 3 else ""

                        personas.append({'Persona_id': persona_id, 'Titulo': titulo, 'Primer_nombre': p_nombre, 'Segundo_nombre': s_nombre, 'Primer_apellido': p_apellido, 'Segundo_apellido': s_apellido, 'Correo': 'N/A'})

                    relacion = {'Persona_id': persona_id, 'Evento_id': ev_id, 'Rol': rol}
                    if relacion not in presentan:
                        presentan.append(relacion)

            procesar_persona(row['Ponente'], 'Ponente', evento_id)
            procesar_persona(row['Maestro de ceremonias'], 'Maestro de ceremonias', evento_id)

        return {
            'DEPARTAMENTO': pd.DataFrame(departamentos),
            'TIPO_ACTIVIDAD': pd.DataFrame(tipos_actividad),
            'EMPRESA_ORG': pd.DataFrame(empresas),
            'EVENTO': pd.DataFrame(eventos).drop_duplicates(),
            'LUGAR': pd.DataFrame(lugares).drop_duplicates(),
            'PERSONA': pd.DataFrame(personas).drop_duplicates(),
            'LIDERA': pd.DataFrame(lidera).drop_duplicates(),
            'SESION': pd.DataFrame(sesiones),
            'PRESENTA': pd.DataFrame(presentan).drop_duplicates()
        }


class DataStorage:
    def __init__(self, tablas_dict, df_rechazos, df_validos):
        self.tablas = tablas_dict
        self.rechazos = df_rechazos
        self.validos = df_validos

    def save_all(self):
        if not os.path.exists(CARPETA_SALIDA):
            os.makedirs(CARPETA_SALIDA)

        print("\n--- Guardando Tablas Relacionales ---")
        for nombre, df in self.tablas.items():
            if df.empty:
                print(f"[OMITIDO] {nombre}.csv (Tabla vacía)")
                continue
            path = f'{CARPETA_SALIDA}/{nombre}.csv'
            df.to_csv(path, index=False, encoding='utf-8-sig')
            print(f"[GUARDADO] {path} ({len(df)} filas)")

        print("\n--- Guardando Registros Rechazados ---")
        if not self.rechazos.empty:
            path_rechazos = f'{CARPETA_SALIDA}/ERRORES_REGISTROS_RECHAZADOS.csv'
            self.rechazos.to_csv(path_rechazos, index=False, encoding='utf-8-sig')
            print(f"[ADVERTENCIA] {path_rechazos} ({len(self.rechazos)} filas con errores)")
        else:
            print("[INFO] No hubo registros rechazados.")

        print("\n--- Guardando Reporte de Auditoría ---")
        if not self.validos.empty and 'Correcciones_Aplicadas' in self.validos.columns:
            corregidos = self.validos[self.validos['Correcciones_Aplicadas'] != "Ninguna"]
            if not corregidos.empty:
                path_correcciones = f'{CARPETA_SALIDA}/AUDITORIA_CORRECCIONES.csv'
                columnas_reporte = ['Actividad', 'Título', 'Ponente', 'Correcciones_Aplicadas']
                corregidos[columnas_reporte].to_csv(path_correcciones, index=False, encoding='utf-8-sig')
                print(f"[GUARDADO] {path_correcciones} ({len(corregidos)} registros corregidos)")
            else:
                print("[INFO] No se requirieron correcciones en este lote.")

        print("\n--- Comprimiendo Archivos ---")
        zip_name = 'Entregable_ETL_DB'
        shutil.make_archive(zip_name, 'zip', CARPETA_SALIDA)
        print(f"[EXITO] Archivo ZIP '{zip_name}.zip' creado exitosamente.")

        try:
            from google.colab import files
            print("Iniciando descarga del archivo ZIP a tu computadora...")
            files.download(f'{zip_name}.zip')
        except ImportError:
            print("[INFO] Entorno local: Revisa la carpeta y el archivo .zip generado.")


# Ejecución completa
def ejecutar_pipeline():

    # 1 y 2. Carga
    print("\n[ETAPA 1 y 2] Cargando datos...")
    loader = ETLDataLoader(RUTA_ARCHIVO_ORIGEN)
    df_raw = loader.load_data()

    # 3. Layout
    print("\n[ETAPA 3] Validando layout y tipos de datos...")
    validator = SchemaValidator(df_raw)
    df_layout_ok = validator.validate_layout()

    # 4 y 5. Limpieza y Validación
    print("\n[ETAPA 4 y 5] Limpiando y validando información...")
    cleaner = DataCleanerAndValidator(df_layout_ok)
    df_valid, df_rejected = cleaner.clean_and_validate()

    # 6 y 7. Transformación
    print("\n[ETAPA 6 y 7] Transformando y modelando tablas (3FN)...")
    transformer = DataTransformer(df_valid)
    tablas_db = transformer.transform()
    for nombre, df in tablas_db.items():
        print(f"   - {nombre}: {len(df)} registros")

    # 8. Almacenamiento
    print("\n[ETAPA 8] Almacenando resultados...")
    storage = DataStorage(tablas_db, df_rejected, df_valid)
    storage.save_all()


# Ejecutar el pipeline
ejecutar_pipeline()


[ETAPA 1 y 2] Cargando datos...
[OK] Carga exitosa. Registros leidos: 294

[ETAPA 3] Validando layout y tipos de datos...
[OK] Layout validado: Todas las columnas requeridas estan presentes.
[OK] Tipos de datos convertidos correctamente (datetime / time).

[ETAPA 4 y 5] Limpiando y validando información...
[OK] Limpieza y Validacion completadas.
   - Registros validos: 293
   - Registros rechazados: 1

[ETAPA 6 y 7] Transformando y modelando tablas (3FN)...
   - DEPARTAMENTO: 4 registros
   - TIPO_ACTIVIDAD: 18 registros
   - EMPRESA_ORG: 78 registros
   - EVENTO: 293 registros
   - LUGAR: 432 registros
   - PERSONA: 364 registros
   - LIDERA: 293 registros
   - SESION: 432 registros
   - PRESENTA: 563 registros

[ETAPA 8] Almacenando resultados...

--- Guardando Tablas Relacionales ---
[GUARDADO] output_db/DEPARTAMENTO.csv (4 filas)
[GUARDADO] output_db/TIPO_ACTIVIDAD.csv (18 filas)
[GUARDADO] output_db/EMPRESA_ORG.csv (78 filas)
[GUARDADO] output_db/EVENTO.csv (293 filas)
[GUARDADO]

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>